[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/LiquidityMgmt/liquidity_monte_carlo_simulation.ipynb)
# Liquidity Management with Monte Carlo Simulations

## Install & Load

In [1]:
import sys
try:
    import pandas as pd, numpy as np, matplotlib.pyplot as plt
except ImportError:
    !pip install pandas numpy matplotlib
    import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("✅ Libraries ready")

## Load Synthetic Data

In [2]:
URL = 'https://raw.githubusercontent.com/VinayaSharada/FinancialAnalyticsCourse/main/LiquidityMgmt/synthetic_liquidity_data.csv'
df = pd.read_csv(URL, parse_dates=['transaction_date'])
print(f"✅ Loaded {len(df):,} rows")
df.head()

## Preprocess

In [3]:
df['net_amount']=np.where(df.transaction_type=='inflow',
                         df.amount_inr,-df.amount_inr)
daily = df.groupby('transaction_date').net_amount.sum()
print(daily.describe())

## Simulation Parameters

In [4]:
SIMS=1000; DAYS=30; INIT=1e7
print(f"Simulations={SIMS}, Days={DAYS}, InitCash=₹{INIT:,.0f}")

## Run Monte Carlo

In [5]:
mean, std = daily.mean(), daily.std()
results = np.zeros((SIMS, DAYS+1))
results[:,0]=INIT
for i in range(SIMS):
    cash=INIT
    for d in range(1,DAYS+1):
        cash+=np.random.normal(mean,std)
        results[i,d]=cash
print("✅ Simulation done")

## Risk Metrics

In [6]:
final=results[:,-1]
vars = {f"VaR_{int(c*100)}%": np.percentile(final,(1-c)*100)
        for c in [0.95,0.99]}
prob_short = (final<0).mean()*100
print(vars, f"Shortfall%={prob_short:.2f}")

## Visualizations

In [7]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,5))
plt.plot(results[:50].T, color='gray', alpha=0.3)
p5,p50,p95 = np.percentile(results, [5,50,95], axis=0)
plt.fill_between(range(DAYS+1), p5,p95, color='red', alpha=0.2)
plt.plot(p50, 'r-')
plt.title('Sample Paths & 90% Band')
plt.show()